In [ ]:


import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import os
import zipfile
import warnings
import community.community_louvain as community_louvain

warnings.filterwarnings("ignore")

plt.rcParams.update({
    'figure.facecolor': 'white',
    'axes.facecolor': 'white',
    'axes.edgecolor': 'black',
    'axes.labelcolor': 'black',
    'xtick.color': 'black',
    'ytick.color': 'black',
    'text.color': 'black'
})

#loading data
file_path = "/content/X_only_all_sheets_cleaned_v3.xlsx"
xls = pd.ExcelFile(file_path)

STAGE_ORDER = [
    'early2cell', 'mid2cell', 'late2cell',
    '4cell', '8cell', '16cell',
    'earlyblastula', 'midblastula', 'lateblastula'
]

stages = [s for s in STAGE_ORDER if s in xls.sheet_names]
print("Stages used:", stages)
#setting pearson coeff
MEAN_EXPR_THRESHOLD = 1.0
CORR_THRESHOLD = 0.85
TOP_HUBS_TO_REPORT = 20
TOP_HUBS_TO_REMOVE = 10


os.makedirs("network_plots", exist_ok=True)
os.makedirs("cytoscape_export", exist_ok=True)


def safe_pct(before, after):
    if pd.isna(before) or before == 0:
        return np.nan
    return ((after - before) / before) * 100.0

def get_largest_cc_size(G):
    if G.number_of_nodes() > 0 and G.number_of_edges() > 0:
        return len(max(nx.connected_components(G), key=len))
    return 0

# ============================================================
# 5. MAIN ANALYSIS
# ============================================================
summary_results = []
all_node_props = []
all_hubs = {}
stage_networks = {}
stage_partitions = {}

for stage in stages:
    print(f"\n--- {stage} ---")

    df = pd.read_excel(xls, sheet_name=stage)
    df.columns = df.columns.astype(str).str.strip()

    if 'Gene' not in df.columns:
        print(f"Skipping {stage}: no Gene column found")
        continue

    df = df.dropna(subset=['Gene'])
    df = df.set_index('Gene')

    # numeric replicate columns only
    numeric_cols = df.select_dtypes(include=np.number).columns.tolist()
    numeric_cols = [c for c in numeric_cols if c != 'is_X']

    if len(numeric_cols) < 2:
        print(f"Skipping {stage}: not enough numeric columns")
        continue

    # collapse duplicate genes if any
    df_expr = df[numeric_cols].groupby(df.index).mean()

    # active genes filter
    df_active = df_expr[df_expr.mean(axis=1) >= MEAN_EXPR_THRESHOLD].copy()
    print("Active genes:", len(df_active))

    if len(df_active) < 5:
        print("Too few active genes, skipping")
        continue

   #network construction
    corr = df_active.T.corr(method='pearson')

    # fix the Gene/Gene reset_index collision
    corr.index.name = 'Source'
    corr.columns.name = 'Target'

    links = corr.stack().reset_index()
    links.columns = ['Source', 'Target', 'Weight']
    links = links.dropna()

    # remove self edges and keep strong positive correlations
    links = links[(links['Source'] != links['Target']) &
                  (links['Weight'] > CORR_THRESHOLD)].copy()

    # remove duplicate undirected edges
    if len(links) > 0:
        links['pair'] = links.apply(lambda x: tuple(sorted([x['Source'], x['Target']])), axis=1)
        links = links.drop_duplicates(subset='pair').drop(columns='pair')

    # make graph
    if len(links) > 0:
        G = nx.from_pandas_edgelist(links, 'Source', 'Target', edge_attr='Weight')
    else:
        G = nx.Graph()

    # add isolated active genes too
    for gene in df_active.index:
        if gene not in G:
            G.add_node(gene)

    if G.number_of_nodes() < 5:
        print("Too few nodes after graph creation, skipping")
        continue

    stage_networks[stage] = G

    # ========================================================
    # COMMUNITY DETECTION
    # ========================================================
    G_conn = G.copy()
    isolated_nodes = [n for n, d in G_conn.degree() if d == 0]
    G_conn.remove_nodes_from(isolated_nodes)

    if G_conn.number_of_nodes() >= 2 and G_conn.number_of_edges() > 0:
        partition_conn = community_louvain.best_partition(G_conn, weight='Weight', random_state=42)
        modularity = community_louvain.modularity(partition_conn, G_conn, weight='Weight')
    else:
        partition_conn = {}
        modularity = np.nan

    full_partition = {gene: -1 for gene in G.nodes()}
    full_partition.update(partition_conn)
    stage_partitions[stage] = full_partition

    num_communities = len(set([v for v in full_partition.values() if v >= 0]))

    #computing network properties
    degree_dict = dict(G.degree())
    density = nx.density(G)
    mean_degree = np.mean(list(degree_dict.values())) if len(degree_dict) > 0 else np.nan
    max_degree = np.max(list(degree_dict.values())) if len(degree_dict) > 0 else np.nan
    n_components = nx.number_connected_components(G)
    largest_cc_before = get_largest_cc_size(G)

    if G_conn.number_of_nodes() > 0 and G_conn.number_of_edges() > 0:
        core_numbers = nx.core_number(G_conn)
        avg_core = np.mean(list(core_numbers.values())) if len(core_numbers) > 0 else np.nan

        clustering_dict = nx.clustering(G_conn, weight='Weight')
        mean_clustering = np.mean(list(clustering_dict.values())) if len(clustering_dict) > 0 else np.nan

        betweenness_dict = nx.betweenness_centrality(G_conn, weight='Weight')
        mean_betweenness = np.mean(list(betweenness_dict.values())) if len(betweenness_dict) > 0 else np.nan
    else:
        core_numbers = {}
        avg_core = np.nan
        clustering_dict = {}
        mean_clustering = np.nan
        betweenness_dict = {}
        mean_betweenness = np.nan

    #hubs
    hub_rows = []
    for gene in G.nodes():
        hub_rows.append({
            'Stage': stage,
            'Gene': gene,
            'Degree': degree_dict.get(gene, 0),
            'Core_Number': core_numbers.get(gene, 0),
            'Betweenness': betweenness_dict.get(gene, 0),
            'Clustering': clustering_dict.get(gene, 0),
            'Community': full_partition.get(gene, -1)
        })

    hub_df = pd.DataFrame(hub_rows)
    hub_df = hub_df.sort_values(
        by=['Degree', 'Core_Number', 'Betweenness'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    top_hubs_df = hub_df.head(TOP_HUBS_TO_REPORT).copy()
    all_hubs[stage] = top_hubs_df
    top_hubs_df.to_csv(f'cytoscape_export/{stage}_top20_hubs.csv', index=False)

    print("Top hubs:", top_hubs_df['Gene'].tolist()[:10])

    # ========================================================
    # NODE PROPERTY SAVE
    # ========================================================
    node_rows = []
    for gene in G.nodes():
        row = {
            'Stage': stage,
            'Gene': gene,
            'Degree': degree_dict.get(gene, 0),
            'Core_Number': core_numbers.get(gene, 0),
            'Betweenness': round(betweenness_dict.get(gene, 0), 6),
            'Clustering': round(clustering_dict.get(gene, 0), 6),
            'Community': full_partition.get(gene, -1),
            'Is_Hub': 'Yes' if gene in top_hubs_df['Gene'].tolist() else 'No'
        }
        node_rows.append(row)
        all_node_props.append(row)

    pd.DataFrame(node_rows).to_csv(f'cytoscape_export/{stage}_nodes.csv', index=False)
    links.to_csv(f'cytoscape_export/{stage}_edges.csv', index=False)

    # ========================================================
    # NETWORK VISUALISATION
    # ========================================================
    fig, ax = plt.subplots(figsize=(13, 10))

    partition = full_partition
    comm_ids = sorted(set([v for v in partition.values() if v >= 0]))
    cmap_comm = plt.cm.get_cmap('tab20', max(len(comm_ids), 1))
    colour_map = {c: cmap_comm(i) for i, c in enumerate(comm_ids)}

    pos = nx.spring_layout(G, seed=42, weight='Weight', k=0.55)

    degrees = dict(G.degree())
    max_deg_plot = max(degrees.values()) if len(degrees) > 0 and max(degrees.values()) > 0 else 1
    node_sizes = [60 + 800 * (degrees[n] / max_deg_plot) for n in G.nodes()]
    node_cols = [colour_map.get(partition.get(n, -1), (0.82, 0.82, 0.82, 1)) for n in G.nodes()]

    edge_ws = [G[u][v]['Weight'] for u, v in G.edges()] if G.number_of_edges() > 0 else []
    max_w = max(edge_ws) if len(edge_ws) > 0 else 1
    edge_widths = [0.2 + 2.2 * (w / max_w) for w in edge_ws] if len(edge_ws) > 0 else []

    if G.number_of_edges() > 0:
        nx.draw_networkx_edges(
            G, pos, ax=ax,
            width=edge_widths,
            edge_color='#bdbdbd',
            alpha=0.35
        )

    nx.draw_networkx_nodes(
        G, pos, ax=ax,
        node_size=node_sizes,
        node_color=node_cols,
        alpha=0.92
    )

    hub_labels = {g: g for g in top_hubs_df['Gene'].tolist() if g in G.nodes()}
    nx.draw_networkx_labels(
        G, pos,
        labels=hub_labels,
        ax=ax,
        font_size=7,
        font_weight='bold'
    )

    legend_handles = []
    for c in comm_ids[:15]:
        count_c = sum(1 for v in partition.values() if v == c)
        legend_handles.append(
            mpatches.Patch(facecolor=colour_map[c], label=f'C{c} ({count_c} genes)')
        )

    if len(comm_ids) > 15:
        legend_handles.append(mpatches.Patch(facecolor='grey', label=f'+ {len(comm_ids)-15} more'))

    if len(legend_handles) > 0:
        ax.legend(handles=legend_handles, loc='upper left', fontsize=6, framealpha=0.85, ncol=2)

    ax.set_title(
        f'{stage}\n'
        f'{G.number_of_nodes()} genes, {G.number_of_edges()} edges | '
        f'Density={density:.4f} | Communities={num_communities}',
        fontsize=11,
        fontweight='bold'
    )
    ax.axis('off')
    plt.tight_layout()
    plt.savefig(f'network_plots/{stage}_network.png', dpi=180, bbox_inches='tight', facecolor='white')
    plt.show()



#top 20 hubs
print("\n\nTOP 20 HUB GENES PER STAGE\n")
for stage in stages:
    if stage in all_hubs:
        print(f"\n===== {stage} =====")
        print(all_hubs[stage][['Gene', 'Degree', 'Core_Number', 'Betweenness', 'Community']].to_string(index=False))


zip_name = 'x_network_analysis_outputs.zip'
with zipfile.ZipFile(zip_name, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in ['network_plots', 'cytoscape_export']:
        for f in os.listdir(folder):
            zf.write(f'{folder}/{f}', f'{folder}/{f}')

    for f in [
        'network_summary_per_stage.csv',
        'network_properties_all_stages.csv',
        'top20_hubs_all_stages.csv',
        'hub_perturbation_summary.csv',
        'network_summary.png',
        'hub_perturbation_comparison.png',
        'hub_perturbation_deltas.png'
    ]:
        if os.path.exists(f):
            zf.write(f, f)

print(f"\nSaved: {zip_name}")


# ============================================================
# FUNCTIONAL ENRICHMENT OF HUB GENES
# FIXED VERSION
# - uses top20_hubs_all_stages.csv
# - mouse only
# - handles 429 rate limits with delay + retry
# - handles missing p-value columns safely
# - runs:
#     1. stage-wise enrichment
#     2. pooled enrichment of all hub genes
#     3. unique-per-stage enrichment
# - saves result tables + barplots
# ============================================================

!pip install -q gseapy pandas matplotlib openpyxl

import os
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gseapy as gp

# ------------------------------------------------------------
# INPUT
# ------------------------------------------------------------
hub_file = "top20_hubs_all_stages.csv"

STAGE_ORDER = [
    'early2cell', 'mid2cell', 'late2cell',
    '4cell', '8cell', '16cell',
    'earlyblastula', 'midblastula', 'lateblastula'
]

GENE_SETS = [
    'GO_Biological_Process_2023',
    'GO_Molecular_Function_2023',
    'GO_Cellular_Component_2023'
]

OUTDIR = "functional_enrichment_fixed"
os.makedirs(OUTDIR, exist_ok=True)

# ------------------------------------------------------------
# LOAD HUB FILE
# ------------------------------------------------------------
hubs_df = pd.read_csv(hub_file)
hubs_df['Stage'] = pd.Categorical(hubs_df['Stage'], categories=STAGE_ORDER, ordered=True)
hubs_df = hubs_df.sort_values(['Stage', 'Degree', 'Core_Number', 'Betweenness'],
                              ascending=[True, False, False, False])

print("Loaded hub file")
print(hubs_df.head())

# ------------------------------------------------------------
# HELPERS
# ------------------------------------------------------------
def pick_pval_col(df):
    for c in ['Adjusted P-value', 'P-value', 'Old P-value']:
        if c in df.columns:
            return c
    return None

def clean_res(res, label_name, label_value, genes_used, geneset_name):
    res = res.copy()
    res[label_name] = label_value
    res['GeneSet'] = geneset_name
    res['Input_Genes'] = " ".join(genes_used)

    if 'Overlap' in res.columns:
        parts = res['Overlap'].astype(str).str.split('/', expand=True)
        if parts.shape[1] == 2:
            res['Overlap_num'] = pd.to_numeric(parts[0], errors='coerce')
            res['Overlap_den'] = pd.to_numeric(parts[1], errors='coerce')

    pcol = pick_pval_col(res)
    if pcol is not None:
        res = res.sort_values(pcol, ascending=True)

    return res

def save_barplot(res, save_path, title_text):
    if res is None or len(res) == 0:
        return

    pcol = pick_pval_col(res)
    if pcol is None:
        return

    topn = res.head(10).copy()
    if len(topn) == 0:
        return

    vals = pd.to_numeric(topn[pcol], errors='coerce').replace(0, 1e-300)
    topn = topn.loc[vals.notna()].copy()
    if len(topn) == 0:
        return

    topn['minus_log10_p'] = -np.log10(pd.to_numeric(topn[pcol], errors='coerce').replace(0, 1e-300))

    plt.figure(figsize=(8, max(4, 0.5 * len(topn))))
    plt.barh(topn['Term'][::-1], topn['minus_log10_p'][::-1], edgecolor='black')
    plt.xlabel(f'-log10({pcol})')
    plt.ylabel('')
    plt.title(title_text)
    plt.tight_layout()
    plt.savefig(save_path, dpi=180, bbox_inches='tight')
    plt.close()

def run_enrichr_safe(gene_list, gene_sets, organism='mouse', max_tries=4, sleep_sec=3):
    """
    returns gseapy enrichr result object or None
    """
    for attempt in range(1, max_tries + 1):
        try:
            enr = gp.enrichr(
                gene_list=gene_list,
                gene_sets=gene_sets,
                organism=organism,
                outdir=None,
                cutoff=1.0
            )
            time.sleep(sleep_sec)
            return enr
        except Exception as e:
            msg = str(e)
            print(f"    try {attempt}/{max_tries} failed -> {msg}")

            # longer wait if rate-limited
            if '429' in msg:
                wait_time = sleep_sec * attempt + 2
                print(f"    waiting {wait_time} sec because of rate limit")
                time.sleep(wait_time)
            else:
                time.sleep(sleep_sec)

    return None

def unique_preserve_order(x):
    seen = set()
    out = []
    for g in x:
        if pd.isna(g):
            continue
        g = str(g)
        if g not in seen:
            seen.add(g)
            out.append(g)
    return out

# ------------------------------------------------------------
# BUILD GENE LISTS
# ------------------------------------------------------------
stage_gene_lists = {}
for stage in STAGE_ORDER:
    sub = hubs_df[hubs_df['Stage'] == stage]
    genes = unique_preserve_order(sub['Gene'].tolist())
    if len(genes) > 0:
        stage_gene_lists[stage] = genes

all_hub_genes = unique_preserve_order(hubs_df['Gene'].tolist())

# unique genes per stage (remove genes appearing in any other stage)
stage_unique_lists = {}
for stage in STAGE_ORDER:
    if stage not in stage_gene_lists:
        continue
    this_set = set(stage_gene_lists[stage])
    other_set = set()
    for other_stage in STAGE_ORDER:
        if other_stage == stage or other_stage not in stage_gene_lists:
            continue
        other_set |= set(stage_gene_lists[other_stage])
    uniq = [g for g in stage_gene_lists[stage] if g in (this_set - other_set)]
    stage_unique_lists[stage] = uniq

# save plain gene lists for copy-paste
gene_list_dir = os.path.join(OUTDIR, "gene_lists")
os.makedirs(gene_list_dir, exist_ok=True)

for stage, genes in stage_gene_lists.items():
    with open(os.path.join(gene_list_dir, f"{stage}_hub_genes.txt"), "w") as f:
        f.write(" ".join(genes))

for stage, genes in stage_unique_lists.items():
    with open(os.path.join(gene_list_dir, f"{stage}_unique_hub_genes.txt"), "w") as f:
        f.write(" ".join(genes))

with open(os.path.join(gene_list_dir, "all_hub_genes_pooled.txt"), "w") as f:
    f.write(" ".join(all_hub_genes))

# ------------------------------------------------------------
# 1) STAGE-WISE ENRICHMENT
# ------------------------------------------------------------
stage_results = []

print("\n================ STAGE-WISE ENRICHMENT ================")
stage_dir_root = os.path.join(OUTDIR, "stage_wise")
os.makedirs(stage_dir_root, exist_ok=True)

for stage in STAGE_ORDER:
    if stage not in stage_gene_lists:
        print(f"\n{stage}: no genes, skipping")
        continue

    genes = stage_gene_lists[stage]
    print(f"\n{stage}: {len(genes)} genes")

    if len(genes) < 3:
        print("  too few genes, skipping")
        continue

    stage_dir = os.path.join(stage_dir_root, stage)
    os.makedirs(stage_dir, exist_ok=True)

    for gs in GENE_SETS:
        print(f"  {gs}")
        enr = run_enrichr_safe(genes, gs, organism='mouse', max_tries=4, sleep_sec=3)

        if enr is None or enr.results is None or len(enr.results) == 0:
            print("    no results")
            continue

        res = clean_res(enr.results, 'Stage', stage, genes, gs)
        res.to_csv(os.path.join(stage_dir, f"{gs}_enrichment.csv"), index=False)
        stage_results.append(res)

        save_barplot(
            res,
            os.path.join(stage_dir, f"{gs}_top10_barplot.png"),
            f"{stage} | {gs}"
        )

# combine stage-wise results
if len(stage_results) > 0:
    stage_results_df = pd.concat(stage_results, ignore_index=True)
    stage_results_df.to_csv(os.path.join(OUTDIR, "stage_wise_all_results.csv"), index=False)

    top_rows = []
    for stage in STAGE_ORDER:
        for gs in GENE_SETS:
            tmp = stage_results_df[(stage_results_df['Stage'] == stage) & (stage_results_df['GeneSet'] == gs)].copy()
            if len(tmp) == 0:
                continue
            pcol = pick_pval_col(tmp)
            if pcol is not None:
                tmp = tmp.sort_values(pcol, ascending=True).head(5)
                top_rows.append(tmp)

    if len(top_rows) > 0:
        stage_top_df = pd.concat(top_rows, ignore_index=True)
        stage_top_df.to_csv(os.path.join(OUTDIR, "stage_wise_top5_terms_summary.csv"), index=False)

# ------------------------------------------------------------
# 2) POOLED ENRICHMENT OF ALL HUB GENES
# ------------------------------------------------------------
print("\n================ POOLED HUB ENRICHMENT ================")
pooled_dir = os.path.join(OUTDIR, "pooled_all_hubs")
os.makedirs(pooled_dir, exist_ok=True)

pooled_results = []

if len(all_hub_genes) >= 3:
    print(f"pooled all hubs: {len(all_hub_genes)} unique genes")
    for gs in GENE_SETS:
        print(f"  {gs}")
        enr = run_enrichr_safe(all_hub_genes, gs, organism='mouse', max_tries=4, sleep_sec=3)

        if enr is None or enr.results is None or len(enr.results) == 0:
            print("    no results")
            continue

        res = clean_res(enr.results, 'Analysis', 'pooled_all_hubs', all_hub_genes, gs)
        res.to_csv(os.path.join(pooled_dir, f"{gs}_enrichment.csv"), index=False)
        pooled_results.append(res)

        save_barplot(
            res,
            os.path.join(pooled_dir, f"{gs}_top10_barplot.png"),
            f"Pooled all hubs | {gs}"
        )

if len(pooled_results) > 0:
    pooled_df = pd.concat(pooled_results, ignore_index=True)
    pooled_df.to_csv(os.path.join(OUTDIR, "pooled_all_hubs_results.csv"), index=False)

# ------------------------------------------------------------
# 3) UNIQUE HUBS PER STAGE ENRICHMENT
# ------------------------------------------------------------
print("\n================ UNIQUE HUBS PER STAGE ================")
unique_dir_root = os.path.join(OUTDIR, "unique_hubs_by_stage")
os.makedirs(unique_dir_root, exist_ok=True)

unique_results = []

for stage in STAGE_ORDER:
    genes = stage_unique_lists.get(stage, [])
    print(f"\n{stage}: {len(genes)} unique genes")

    if len(genes) < 3:
        print("  too few unique genes, skipping")
        continue

    stage_dir = os.path.join(unique_dir_root, stage)
    os.makedirs(stage_dir, exist_ok=True)

    for gs in GENE_SETS:
        print(f"  {gs}")
        enr = run_enrichr_safe(genes, gs, organism='mouse', max_tries=4, sleep_sec=3)

        if enr is None or enr.results is None or len(enr.results) == 0:
            print("    no results")
            continue

        res = clean_res(enr.results, 'Stage', stage, genes, gs)
        res.to_csv(os.path.join(stage_dir, f"{gs}_enrichment.csv"), index=False)
        unique_results.append(res)

        save_barplot(
            res,
            os.path.join(stage_dir, f"{gs}_top10_barplot.png"),
            f"{stage} unique hubs | {gs}"
        )

if len(unique_results) > 0:
    unique_df = pd.concat(unique_results, ignore_index=True)
    unique_df.to_csv(os.path.join(OUTDIR, "unique_hubs_stage_results.csv"), index=False)

# ------------------------------------------------------------
# PRINT QUICK SUMMARIES
# ------------------------------------------------------------
print("\n================ QUICK SUMMARY ================")

if len(stage_results) > 0:
    print("\nTop stage-wise terms:")
    tmp = pd.concat(stage_results, ignore_index=True)
    pcol = pick_pval_col(tmp)
    if pcol is not None:
        show_rows = []
        for stage in STAGE_ORDER:
            st = tmp[tmp['Stage'] == stage].copy()
            if len(st) == 0:
                continue
            st = st.sort_values(pcol, ascending=True).head(3)
            show_rows.append(st)
        if len(show_rows) > 0:
            show_df = pd.concat(show_rows, ignore_index=True)
            print(show_df[['Stage', 'GeneSet', 'Term', pcol]].to_string(index=False))

if len(pooled_results) > 0:
    print("\nTop pooled terms:")
    tmp = pd.concat(pooled_results, ignore_index=True)
    pcol = pick_pval_col(tmp)
    if pcol is not None:
        print(tmp.sort_values(pcol, ascending=True)[['Analysis', 'GeneSet', 'Term', pcol]].head(10).to_string(index=False))

print(f"\nDone. All outputs saved in: {OUTDIR}")